
 **RAG-Augmented Multi-Model Approach**

This notebook builds a price-per-square-meter prediction model for Egyptian real estate using:



##  Setup

In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("sentence-transformers", "faiss-cpu", "xgboost", "lightgbm", "catboost", "shap", "plotly")

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import shap

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Color palette
PALETTE = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#7C3AED', '#0891B2']
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams['figure.dpi'] = 120

print('All libraries loaded')

##  Load Data

In [ ]:
# Works on both Kaggle and locally
KAGGLE_PATH = '/kaggle/input/propertyfinder-egypt/propertyfinder_egypt.csv'
LOCAL_PATH  = '..\data\propertyfinder.csv'
DATA_PATH   = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH

df_raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig', low_memory=False)
print(f'Rows: {len(df_raw):,} | Columns: {df_raw.shape[1]}')
df_raw.head(2)

##  Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))


cat_counts = df_raw['category'].value_counts()
axes[0].pie(cat_counts, labels=cat_counts.index.str.title(), autopct='%1.1f%%',
            colors=PALETTE[:2], startangle=90)
axes[0].set_title('Buy vs Rent Split', fontsize=14, fontweight='bold')

top_types = df_raw['property_type'].value_counts().head(8)
axes[1].barh(top_types.index[::-1], top_types.values[::-1], color=PALETTE[0])
axes[1].set_title('Property Types', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
city_counts = df_raw['city'].value_counts().head(8)
fig = px.bar(x=city_counts.values, y=city_counts.index, orientation='h',
             color=city_counts.values, color_continuous_scale='Blues',
             title='Listings by City', labels={'x': 'Count', 'y': 'City'})
fig.update_layout(showlegend=False, height=400)
fig.show()

In [ ]:
# Price Distribution (Buy vs Rent)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

buy_prices  = df_raw[(df_raw['category']=='buy')  & (df_raw['price_egp'] < 5e7)]['price_egp'] / 1e6
rent_prices = df_raw[(df_raw['category']=='rent') & (df_raw['price_egp'] < 5e5)]['price_egp'] / 1e3

axes[0].hist(buy_prices, bins=60, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[0].set_title('Buy Price Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (Million EGP)')

axes[1].hist(rent_prices, bins=60, color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[1].set_title('Rent Price Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Price (Thousand EGP)')

plt.tight_layout()
plt.show()

In [ ]:
#  Geographic scatter 
sample = df_raw.dropna(subset=['lat','lon']).sample(min(5000, len(df_raw)), random_state=42)
fig = px.scatter_mapbox(
    sample, lat='lat', lon='lon',
    color='category', hover_data=['property_type','city','price_egp','area_value'],
    color_discrete_map={'buy': '#2563EB', 'rent': '#16A34A'},
    zoom=5.5, height=520,
    mapbox_style='open-street-map',
    title='Property Listings Map — Egypt'
)
fig.show()

In [ ]:
# Price vs Area scatter
plot_df = df_raw[(df_raw['area_value'] > 0) & (df_raw['area_value'] < 1000)
                 & (df_raw['price_egp'] > 0) & (df_raw['price_egp'] < 8e7)].sample(3000, random_state=42)
fig = px.scatter(
    plot_df, x='area_value', y='price_egp',
    color='category', opacity=0.5, trendline='ols',
    labels={'area_value': 'Area (sqm)', 'price_egp': 'Price (EGP)'},
    title='Price vs Area',
    color_discrete_map={'buy': '#2563EB', 'rent': '#16A34A'}
)
fig.show()

##  Feature Engineering

In [ ]:
df = df_raw.copy()

# price per sqm
df['price_per_sqm'] = df['price_egp'] / df['area_value']

# filter to BUY only 
df = df[df['category'] == 'buy'].copy()

# removing outliers
lo, hi = df['price_per_sqm'].quantile([0.01, 0.99])
df = df[(df['price_per_sqm'] >= lo) & (df['price_per_sqm'] <= hi)]
df = df[(df['area_value'] > 20) & (df['area_value'] < 2000)]

print(f'After filtering: {len(df):,} rows')
print(f'Price per sqm range: {df["price_per_sqm"].min():,.0f} – {df["price_per_sqm"].max():,.0f} EGP/sqm')

In [ ]:
# Amenity features
def has_amenity(amenities_str, keyword):
    return int(str(amenities_str).lower().find(keyword.lower()) >= 0)


df['amenity_count']    = df['amenities'].apply(lambda x: len(str(x).split('|')) if pd.notna(x) and x!='' else 0)
df['has_pool']         = df['amenities'].apply(lambda x: has_amenity(x, 'Pool'))
df['has_gym']          = df['amenities'].apply(lambda x: has_amenity(x, 'Gym'))
df['has_security']     = df['amenities'].apply(lambda x: has_amenity(x, 'Security'))
df['has_parking']      = df['amenities'].apply(lambda x: has_amenity(x, 'Parking'))
df['has_garden']       = df['amenities'].apply(lambda x: has_amenity(x, 'Garden'))
df['has_balcony']      = df['amenities'].apply(lambda x: has_amenity(x, 'Balcony'))
df['has_private_pool'] = df['amenities'].apply(lambda x: has_amenity(x, 'Private Pool'))
df['has_spa']          = df['amenities'].apply(lambda x: has_amenity(x, 'Spa'))
df['has_ac']           = df['amenities'].apply(lambda x: has_amenity(x, 'A/C'))

print(' Amenity features created')

In [ ]:
df['listed_date'] = pd.to_datetime(df['listed_date'], errors='coerce', utc=True)
reference_date = pd.Timestamp('2026-03-05', tz='UTC')
df['days_since_listed'] = (reference_date - df['listed_date']).dt.days.fillna(-1)

df['bedrooms']  = pd.to_numeric(df['bedrooms'],  errors='coerce').fillna(0)
df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce').fillna(0)
df['bed_bath_ratio'] = (df['bedrooms'] / df['bathrooms'].replace(0, 1)).round(2)
df['total_rooms'] = df['bedrooms'] + df['bathrooms']

df['is_furnished'] = (df['furnished'].str.upper() == 'YES').astype(int)

status_map = {'completed': 2, 'completed_primary': 2, 'off_plan': 0, 'off_plan_primary': 0}
df['completion_score'] = df['completion_status'].map(status_map).fillna(1)

df['desc_length'] = df['description'].fillna('').apply(len)

print(' Date, spec, and text-length features created')

In [ ]:
# used median price_per_sqm per category to avoid target leakage with smoothing)
global_mean = df['price_per_sqm'].mean()

def target_encode(df, col, target='price_per_sqm', smoothing=10):
    stats  = df.groupby(col)[target].agg(['mean', 'count'])
    smooth = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    return df[col].map(smooth).fillna(global_mean)

df['city_encoded']         = target_encode(df, 'city')
df['town_encoded']         = target_encode(df, 'town')
df['district_encoded']     = target_encode(df, 'district')
df['property_type_encoded']= target_encode(df, 'property_type')

print('Target encoding done')

In [ ]:
# feature correlation with target
numeric_feats = ['area_value','bedrooms','bathrooms','amenity_count','has_pool','has_gym',
                 'has_security','has_parking','has_private_pool','is_furnished',
                 'city_encoded','district_encoded','property_type_encoded','desc_length',
                 'days_since_listed','completion_score','price_per_sqm']

corr = df[numeric_feats].corr()['price_per_sqm'].drop('price_per_sqm').sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
colors = [PALETTE[2] if v < 0 else PALETTE[0] for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Price per sqm', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

##  RAG — Retrieval-Augmented Feature

We embed each listing's description using `sentence-transformers`, build a FAISS nearest-neighbor index, then for each property retrieve its **K=5 most similar listings** by description semantics. The **average price_per_sqm of those neighbors** becomes a powerful feature — essentially: *"similar-described properties cost X per sqm"*.

This is a form of **Retrieval-Augmented Prediction**: use retrieved context to improve a downstream model.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

print('Loading sentence-transformer model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')  

descriptions = df['description'].fillna('').tolist()
print(f'Encoding {len(descriptions):,} descriptions...')
embeddings = embedder.encode(descriptions, batch_size=256, show_progress_bar=True,
                              convert_to_numpy=True, normalize_embeddings=True)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  
index.add(embeddings.astype('float32'))
print(f'FAISS index built with {index.ntotal:,} vectors')

In [ ]:
# K nearest neighbors 
K = 6  
prices = df['price_per_sqm'].values.astype('float32')

_, indices = index.search(embeddings.astype('float32'), K)

rag_prices = []
for i, neighbors in enumerate(indices):
    neighbor_prices = [prices[j] for j in neighbors[1:] if j != i]
    rag_prices.append(np.mean(neighbor_prices) if neighbor_prices else prices[i])

df['rag_neighbor_price_per_sqm'] = rag_prices

# correlation check
corr_rag = df['rag_neighbor_price_per_sqm'].corr(df['price_per_sqm'])
print(f' RAG feature created | Correlation with target: {corr_rag:.4f}')

##  Model Training

In [ ]:
FEATURES = [
    'area_value', 'bedrooms', 'bathrooms', 'bed_bath_ratio', 'total_rooms',
    'lat', 'lon', 'city_encoded', 'town_encoded', 'district_encoded',
    'property_type_encoded', 'completion_score', 'is_furnished',
    'amenity_count', 'has_pool', 'has_gym', 'has_security', 'has_parking',
    'has_garden', 'has_balcony', 'has_private_pool', 'has_spa', 'has_ac',
    'days_since_listed', 'desc_length',
    'rag_neighbor_price_per_sqm',
]

TARGET = 'price_per_sqm'

model_df = df[FEATURES + [TARGET]].dropna()
X = model_df[FEATURES]
y = np.log1p(model_df[TARGET])  # for better normality

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Features: {len(FEATURES)}')

In [ ]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    rmse  = np.sqrt(mean_squared_error(y_te, preds))
    mae   = mean_absolute_error(y_te, preds)
    r2    = r2_score(y_te, preds)
    # back-transform RMSE to original scale for intuition
    rmse_orig = np.sqrt(mean_squared_error(np.expm1(y_te), np.expm1(preds)))
    print(f'  {name:<28} RMSE(log)={rmse:.4f}  MAE(log)={mae:.4f}  R²={r2:.4f}  RMSE(EGP/sqm)={rmse_orig:,.0f}')
    return {'Model': name, 'RMSE_log': rmse, 'MAE_log': mae, 'R2': r2, 'RMSE_orig': rmse_orig, 'obj': model}

results = []
print('Training models...\n')

In [ ]:
# 1. ridge Regression (baseline — needs scaled data)
results.append(evaluate('Ridge Regression', Ridge(alpha=10),
                         X_train_sc, X_test_sc, y_train, y_test))

In [ ]:
# 2. random forest
results.append(evaluate('Random Forest', RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=42),
                         X_train, X_test, y_train, y_test))

In [ ]:
# 3. XGBoost
results.append(evaluate('XGBoost', xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=7,
                                                      subsample=0.8, colsample_bytree=0.8, n_jobs=-1,
                                                      random_state=42, verbosity=0),
                         X_train, X_test, y_train, y_test))

In [ ]:
# 4. LightGBM
results.append(evaluate('LightGBM', lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63,
                                                        feature_fraction=0.8, bagging_fraction=0.8,
                                                        bagging_freq=5, n_jobs=-1, random_state=42, verbose=-1),
                         X_train, X_test, y_train, y_test))

In [ ]:
# 5. CatBoost
results.append(evaluate('CatBoost', CatBoostRegressor(iterations=500, learning_rate=0.05, depth=8,
                                                        loss_function='RMSE', random_seed=42, verbose=False),
                         X_train, X_test, y_train, y_test))

In [ ]:
# 6. Stacking Ensemble
print('  Training Stacking Ensemble (this takes a moment)...')
estimators = [
    ('rf',   RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)),
    ('xgb',  xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, n_jobs=-1, random_state=42, verbosity=0)),
    ('lgbm', lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, n_jobs=-1, random_state=42, verbose=-1)),
    ('cat',  CatBoostRegressor(iterations=300, learning_rate=0.05, random_seed=42, verbose=False)),
]
stacker = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=1), cv=5, n_jobs=-1)
results.append(evaluate('Stacking Ensemble', stacker, X_train, X_test, y_train, y_test))

In [ ]:
# comparison
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'obj'} for r in results])
results_df = results_df.sort_values('R2', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = [PALETTE[0] if i == 0 else '#94A3B8' for i in range(len(results_df))]
axes[0].barh(results_df['Model'][::-1], results_df['R2'][::-1], color=colors[::-1])
axes[0].set_title('R² Score (higher = better)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('R²')

axes[1].barh(results_df['Model'][::-1], results_df['RMSE_orig'][::-1], color=colors[::-1])
axes[1].set_title('RMSE (lower = better)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('RMSE (EGP/sqm)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

print('\n Full comparison:')
display(results_df.drop(columns=['RMSE_log','MAE_log']).rename(columns={'RMSE_orig':'RMSE (EGP/sqm)'}))

##  SHAP Feature Importance (Best Model)

In [ ]:
# Use LightGBM for SHAP (fastest + most interpretable)
best_lgbm = [r['obj'] for r in results if 'LightGBM' in r['Model']][0]

explainer = shap.TreeExplainer(best_lgbm)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, show=False, max_display=20)
plt.title('SHAP Feature Importance — LightGBM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# bar plot of mean SHAP
mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=FEATURES).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(mean_shap.index[::-1], mean_shap.values[::-1], color=PALETTE[0])
ax.set_title('Top 15 Features by Mean SHAP', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean SHAP value')
plt.tight_layout()
plt.show()

##  Predictions vs Actual

In [ ]:
best_model = [r for r in results if r['R2'] == results_df['R2'].max()][0]
y_pred_log = best_model['obj'].predict(X_test)
y_pred = np.expm1(y_pred_log)
y_actual = np.expm1(y_test)

fig = px.scatter(
    x=y_actual, y=y_pred, opacity=0.4,
    labels={'x': 'Actual (EGP/sqm)', 'y': 'Predicted (EGP/sqm)'},
    title=f'Predicted vs Actual — {best_model["Model"]}',
    color_discrete_sequence=[PALETTE[0]]
)
max_val = max(y_actual.max(), y_pred.max())
fig.add_shape(type='line', x0=0, y0=0, x1=max_val, y1=max_val,
              line=dict(color='red', dash='dash', width=2))
fig.show()

print(f'Best model: {best_model["Model"]}')
print(f'R²:   {best_model["R2"]:.4f}')
print(f'RMSE: {best_model["RMSE_orig"]:,.0f} EGP/sqm')